# Mode D: Adaptive Heuristics

**NSGA-II + Adaptive Selection** - Learns which heuristics work best and adapts probabilities.

## 1. Imports

In [ ]:
from __future__ import annotations
import random, copy, time
import numpy as np
from pathlib import Path
from datetime import datetime

from deap import base, creator, tools
from schedule_engine.notebooks.core import (
    load_data, create_random_individual, course_aware_crossover, smart_mutation,
    create_evaluator, get_constraint_breakdown, setup_deap, get_best_individual, 
    EvolutionStats, print_constraint_details
)
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary
from schedule_engine.notebooks.strategies import AdaptiveSelector

print("✅ Imports successful")

## 2. Mode D Configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters
POP_SIZE = 50
NGEN = 100
CXPB = 0.9
MUTPB = 0.2
FITNESS_WEIGHTS = (-1.0, -1.0)  # Equal weight for hard & soft

# MODE D: Adaptive selection
REPAIR_PROB = 0.3
LEARNING_RATE = 0.1
MIN_PROB = 0.05

# Paths
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_d_adaptive/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Mode D: pop={POP_SIZE}, ngen={NGEN}, learning_rate={LEARNING_RATE}")

## 3. Load Data

In [ ]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)
evaluate = create_evaluator(data)

print(f" {data.summary()}")

## 4. Test Adaptive Selector

In [ ]:
# Test adaptive selector
selector = AdaptiveSelector(learning_rate=LEARNING_RATE, min_prob=MIN_PROB)
test_ind = create_random_individual(data)

print(f"Initial probs: {selector.probs}")
print(f"Initial fitness: hard={evaluate(test_ind)[0]}")

for _ in range(10):
    name, fixes = selector.apply(test_ind, data)
    
print(f"After 10 applications:")
print(f"  Fitness: hard={evaluate(test_ind)[0]}")
print(f"  Probs: {selector.probs}")

## 5. Adaptive NSGA-II Evolution (Mode D)

In [ ]:
LOG_INTERVAL = 10  # Show detailed breakdown every N gens

def run_adaptive_nsga2():
    """Run NSGA-II with adaptive heuristic selection."""
    print(f"🧠 Adaptive NSGA-II: pop={POP_SIZE}, ngen={NGEN}")
    start = time.time()
    
    setup_deap(FITNESS_WEIGHTS)
    selector = AdaptiveSelector(learning_rate=LEARNING_RATE, min_prob=MIN_PROB)
    
    toolbox = base.Toolbox()
    toolbox.register("individual", lambda: creator.Individual(create_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    total_repairs = 0
    prob_history = []  # Track probability evolution
    
    for gen in range(NGEN):
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # === MODE D SPECIFIC: Adaptive Repair ===
        for ind in offspring:
            if random.random() < REPAIR_PROB:
                genes = list(ind)
                _, fixes = selector.apply(genes, data)
                total_repairs += fixes
                ind[:] = genes
                del ind.fitness.values
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        prob_history.append(dict(selector.probs))
        
        # Show detailed breakdown every LOG_INTERVAL gens
        if gen % LOG_INTERVAL == 0 or gen == NGEN - 1:
            best_ind = min(pop, key=lambda ind: (ind.fitness.values[0], ind.fitness.values[1]))
            breakdown = get_constraint_breakdown(list(best_ind), data)
            hard_names = {'student_group_exclusivity', 'instructor_exclusivity', 'instructor_qualifications', 
                         'room_suitability', 'room_exclusivity', 'instructor_time_availability', 
                         'room_time_availability', 'course_completeness'}
            hard_bd = {k: v for k, v in breakdown.items() if k in hard_names}
            soft_bd = {k: v for k, v in breakdown.items() if k not in hard_names}
            print_constraint_details(hard_bd, soft_bd, gen)
            probs_str = ", ".join(f"{k}:{v:.2f}" for k, v in selector.probs.items())
            print(f"       Adaptive probs: [{probs_str}]")
    
    stats.elapsed_time = time.time() - start
    print(f"✅ Done in {stats.elapsed_time:.1f}s (total repairs: {total_repairs})")
    print(f"Final heuristic stats: {selector.get_stats()}")
    return pop, stats, prob_history

final_pop, stats, prob_history = run_adaptive_nsga2()

## 6. Results

In [ ]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_d_convergence.png", title_prefix="Mode D: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_d_breakdown.png", title="Mode D: Constraint Violations")

## 7. Adaptive Probability Evolution

In [ ]:
import matplotlib.pyplot as plt

# Plot probability evolution
fig, ax = plt.subplots(figsize=(10, 5))

heuristic_names = list(prob_history[0].keys())
for name in heuristic_names:
    probs = [p[name] for p in prob_history]
    ax.plot(probs, label=name)

ax.set_xlabel("Generation")
ax.set_ylabel("Selection Probability")
ax.set_title("Mode D: Adaptive Heuristic Probability Evolution")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mode_d_probabilities.png", dpi=150)
plt.show()

## 8. Export Results

In [ ]:
from schedule_engine.notebooks.export import export_full_results

export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_d_adaptive",
)

print(f"\n All files saved to: {OUTPUT_DIR}")